#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
# @title ⚙️ 1. Install Dependencies
# @markdown Installs Hugging Face Transformers, SoundFile, Einops, and Zyphra's prebuilt text normalizer wheel.

from IPython.display import clear_output

print("🚀 [@CoinNoin] Installing dependencies for Zyphra/ZONOS3-Mini-hf...")
!pip install -q "transformers>=4.57" soundfile einops zyphra-textnorm

clear_output()
print("✅ [@CoinNoin] Dependencies installed successfully!")

In [ ]:
# @title 🧠 2. Load ZONOS3-Mini Model & Built-in Voices
# @markdown Loads `Zyphra/ZONOS3-Mini-hf` directly in PyTorch on your Colab GPU and fetches the 27 precomputed built-in voice embeddings.

import os
import glob
import inspect
import numpy as np
import torch
from huggingface_hub import snapshot_download
from transformers import AutoModel
from IPython.display import clear_output

MODEL_ID = "Zyphra/ZONOS3-Mini-hf"
VOICES_REPO = "Zyphra/ZONOS3-Mini"
VOICES_ROOT = "/content/zonos3_voices"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ [@CoinNoin] Loading {MODEL_ID} on {device.upper()}...")

model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
model = model.to(device).eval()

print("🎙️ [@CoinNoin] Downloading 27 built-in voice embeddings (.embedding.npy)...")
snapshot_download(
    repo_id=VOICES_REPO,
    local_dir=VOICES_ROOT,
    allow_patterns=["gguf/voices/**/*.embedding.npy", "gguf/voices/*.embedding.npy"],
)

# Index all downloaded built-in voice embeddings
VOICE_MAP = {}
for path in sorted(glob.glob(os.path.join(VOICES_ROOT, "gguf", "voices", "**", "*.embedding.npy"), recursive=True)):
    short_name = os.path.basename(path).replace(".embedding.npy", "")
    VOICE_MAP[short_name] = path


def build_generate_kwargs(model_obj, raw_kwargs):
    """Map parameter aliases and keep only arguments supported by model.generate."""
    sig = inspect.signature(model_obj.generate)
    params = sig.parameters
    has_var_kw = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values())

    aliases = {
        "top_k": ("top_k", "topk"),
        "max_tokens": ("max_new_tokens", "max_tokens"),
    }

    resolved = {}
    for k, v in raw_kwargs.items():
        if v is None:
            continue
        if k in aliases:
            for candidate in aliases[k]:
                if candidate in params:
                    resolved[candidate] = v
                    break
        elif has_var_kw or k in params:
            resolved[k] = v
    return resolved


def unpack_audio(result, model_obj):
    """Extract (1D float32 numpy waveform, sample_rate) from model.generate output."""
    default_sr = (
        getattr(model_obj, "sample_rate", None)
        or getattr(model_obj, "sampling_rate", None)
        or getattr(getattr(model_obj, "config", None), "sampling_rate", None)
        or getattr(getattr(model_obj, "config", None), "sample_rate", None)
        or 44100
    )
    sr = int(default_sr)
    audio = result

    if isinstance(result, tuple) and len(result) == 2:
        a, b = result
        if isinstance(b, (int, float)):
            audio, sr = a, int(b)
        elif isinstance(a, (int, float)):
            sr, audio = int(a), b
    elif isinstance(result, dict):
        for key in ("audio", "waveform", "waveforms", "wav"):
            if key in result:
                audio = result[key]
                break
        for sr_key in ("sample_rate", "sampling_rate", "sr"):
            if sr_key in result and result[sr_key] is not None:
                sr = int(result[sr_key])
                break
    else:
        for attr in ("audio", "waveform", "waveforms", "wav"):
            if hasattr(result, attr):
                audio = getattr(result, attr)
                break
        for sr_attr in ("sample_rate", "sampling_rate", "sr"):
            if hasattr(result, sr_attr) and getattr(result, sr_attr) is not None:
                sr = int(getattr(result, sr_attr))
                break

    if isinstance(audio, torch.Tensor):
        audio = audio.detach().float().cpu().numpy()
    else:
        audio = np.asarray(audio, dtype=np.float32)

    audio = np.squeeze(audio)
    if audio.ndim > 1:
        audio = audio[0]
    return audio.astype(np.float32), sr


clear_output()
gpu_label = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"✅ [@CoinNoin] {MODEL_ID} loaded successfully on {gpu_label}!")
print(f"🔎 model.generate signature: {inspect.signature(model.generate)}")
print(f"\n🎙️ Built-in voices loaded ({len(VOICE_MAP)}):")
for vname in VOICE_MAP:
    print(f"   • {vname}")

In [ ]:
# @title 🗣️ 3. Standard Text-to-Speech (All 43 Languages)
# @markdown Enter text in any of the 43 supported languages and choose a built-in voice printed in Cell 2 (or leave as `auto` / `none`).

target_text = "Hello! This is ZONOS3-Mini running directly in Google Colab." # @param {type:"string"}
voice_name = "Meilin_zh-energetic_F_1" # @param ["none", "auto", "Cillian_en-Irish_M_1", "Imogen_en-British_F_1", "Jett_en-energetic_M_1", "Morgana_en-witch_F_1", "Rupert_en-British_M_1", "Sakura_en-anime_F_1", "Savannah_en-American_F_1", "Thaddeus_en-announcervoice_M_1", "Willow_en-asrm_F_1", "Wyatt_en-American_M_1", "Gaspard_fr_M_1", "Liesel_de_F_1", "Wolfgang_de_M_1", "Ananya_hi_F_1", "Arjun_hi_M_1", "Ginevra_it_F_1", "Lorenzo_it_M_1", "Minjun_ko_M_1", "Seoyeon_ko_F_1", "Haoran_zh-energetic_M_1", "Meilin_zh-energetic_F_1", "Leonor_pt_F_1", "Thiago_pt_M_1", "Rodrigo_es_M_1", "Ximena_es_F_1", "Chaitanya_te_M_1", "Sravya_te_F_1"]
# @markdown *Type `auto` for the first built-in voice, `none` for default unconditioned voice, or a voice name from Cell 2 (e.g., `Arjun`, `Cillian`, `Jett`).*

language = "en_us (English)" # @param ["en_us (English)", "fr_fr (French)", "es (Spanish)", "de (German)", "it (Italian)", "pt_br (Portuguese)", "nl (Dutch)", "el (Greek)", "sv (Swedish)", "da (Danish)", "no (Norwegian)", "fi (Finnish)", "pl (Polish)", "cs (Czech)", "sk (Slovak)", "hu (Hungarian)", "ro (Romanian)", "bg (Bulgarian)", "ru (Russian)", "uk (Ukrainian)", "tr (Turkish)", "ar (Arabic)", "he (Hebrew)", "ka (Georgian)", "cmn (Mandarin Chinese)", "ja (Japanese)", "ko (Korean)", "vi (Vietnamese)", "id (Indonesian)", "ms (Malay)", "th (Thai)", "tl (Tagalog)", "hi (Hindi)", "bn (Bengali)", "ta (Tamil)", "te (Telugu)", "mr (Marathi)", "gu (Gujarati)", "kn (Kannada)", "ml (Malayalam)", "pa (Punjabi)", "or (Odia)", "ur (Urdu)"]

# @markdown ---
# @markdown **Generation Parameters:**
temperature = 0.75 # @param {type:"slider", min:0.1, max:1.5, step:0.05}
top_p = 0.69 # @param {type:"slider", min:0.1, max:1.0, step:0.01}
top_k = 80 # @param {type:"slider", min:0, max:200, step:5}
repetition_penalty = 2.4 # @param {type:"slider", min:1.0, max:4.0, step:0.1}
max_new_frames = 450 # @param {type:"slider", min:75, max:1500, step:75}
# @markdown *15 frames = 1 second of audio (450 frames = up to 30 seconds).*
seed = -1 # @param {type:"integer"}
# @markdown *Set `seed` to `-1` for a random seed, or any non-negative integer for reproducible output.*

import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display

lang_code = language.split(" ")[0].strip()

# Resolve selected built-in speaker embedding
speaker_emb_np = None
selected_label = "Default (No Embedding)"

q = voice_name.strip().lower()
if q not in ("none", "default"):
    if q in ("auto", ""):
        selected_label = next(iter(VOICE_MAP.keys()))
        speaker_emb_np = np.load(VOICE_MAP[selected_label]).astype(np.float32)
    else:
        matched = [name for name in VOICE_MAP if q == name.lower() or q in name.lower()]
        if not matched:
            raise ValueError(f"❌ Voice '{voice_name}' not found. Available: {list(VOICE_MAP.keys())}")
        selected_label = matched[0]
        speaker_emb_np = np.load(VOICE_MAP[selected_label]).astype(np.float32)

base_kwargs = dict(
    temperature=float(temperature),
    topk=int(top_k),
    top_p=float(top_p),
    repetition_penalty=float(repetition_penalty),
    max_new_frames=int(max_new_frames),
    text_normalization=True,
)
if seed >= 0:
    base_kwargs["seed"] = int(seed)
if speaker_emb_np is not None:
    base_kwargs["speaker_embedding"] = torch.from_numpy(speaker_emb_np).to(device)


def run_tts_generate(text_input, lang_val, extra_kwargs):
    try:
        return model.generate(text_input, language=lang_val, **extra_kwargs)
    except (TypeError, ValueError, KeyError, RuntimeError):
        # Fallback if a specific language code isn't mapped in the runtime/normalizer
        short_lang = lang_val.split("_")[0]
        if short_lang != lang_val:
            try:
                return model.generate(text_input, language=short_lang, **extra_kwargs)
            except Exception:
                pass
        fallback_kwargs = {**extra_kwargs, "text_normalization": False}
        try:
            return model.generate(text_input, language="en_us", **fallback_kwargs)
        except Exception:
            fallback_kwargs.pop("text_normalization", None)
            return model.generate(text_input, **fallback_kwargs)


print(f"🎙️ [@CoinNoin] Generating speech ({language}) with voice: {selected_label}...")

with torch.inference_mode():
    result = run_tts_generate(target_text, lang_code, base_kwargs)

audio, sr = unpack_audio(result, model)
output_path = "output_tts_ZONOS3-Mini_CoinNoin.wav"
sf.write(output_path, audio, sr)

print(f"✅ Generation complete! Saved to {output_path} ({sr} Hz)")
display(Audio(output_path))

In [ ]:
# @title 🎭 4. Zero-Shot Voice Cloning (All 43 Languages)
# @markdown Run this cell and upload a clean reference audio file (`.wav`, `.mp3`, or `.flac`, 5–30 seconds). **No transcript is needed.**

target_text = "This is a zero-shot voice cloning test using ZONOS3-Mini directly on Google Colab." # @param {type:"string"}
language = "en_us (English)" # @param ["en_us (English)", "fr_fr (French)", "es (Spanish)", "de (German)", "it (Italian)", "pt_br (Portuguese)", "nl (Dutch)", "el (Greek)", "sv (Swedish)", "da (Danish)", "no (Norwegian)", "fi (Finnish)", "pl (Polish)", "cs (Czech)", "sk (Slovak)", "hu (Hungarian)", "ro (Romanian)", "bg (Bulgarian)", "ru (Russian)", "uk (Ukrainian)", "tr (Turkish)", "ar (Arabic)", "he (Hebrew)", "ka (Georgian)", "cmn (Mandarin Chinese)", "ja (Japanese)", "ko (Korean)", "vi (Vietnamese)", "id (Indonesian)", "ms (Malay)", "th (Thai)", "tl (Tagalog)", "hi (Hindi)", "bn (Bengali)", "ta (Tamil)", "te (Telugu)", "mr (Marathi)", "gu (Gujarati)", "kn (Kannada)", "ml (Malayalam)", "pa (Punjabi)", "or (Odia)", "ur (Urdu)"]
speaker_mode = "embedding"
speaker_cfg_scale = 1.2 # @param {type:"slider", min:1.0, max:2.5, step:0.1}
# @markdown *`1.0` = off; increase to `1.2–1.8` for stronger reference adherence.*

# @markdown ---
# @markdown **Generation Parameters:**
temperature = 0.75 # @param {type:"slider", min:0.1, max:1.5, step:0.05}
top_p = 0.69 # @param {type:"slider", min:0.1, max:1.0, step:0.01}
top_k = 80 # @param {type:"slider", min:0, max:200, step:5}
repetition_penalty = 2.4 # @param {type:"slider", min:1.0, max:4.0, step:0.1}
max_new_frames = 450 # @param {type:"slider", min:75, max:1500, step:75}
# @markdown *15 frames = 1 second of audio (450 frames = up to 30 seconds).*
seed = -1 # @param {type:"integer"}

import os
import soundfile as sf
import torch
import torchaudio
from google.colab import files
from IPython.display import Audio, display, clear_output

lang_code = language.split(" ")[0].strip()

print("📤 Please upload your reference audio file (.wav / .mp3 / .flac, 5–30s):")
uploaded = files.upload()

if len(uploaded) == 0:
    print("❌ No file uploaded. Please run the cell again and select a file.")
else:
    ref_path = list(uploaded.keys())[0]
    clear_output()
    print(f"✅ Uploaded '{ref_path}'. Encoding reference & generating cloned speech ({language})...")

    try:
        wav, ref_sr = torchaudio.load(ref_path)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        wav = wav.squeeze(0)

        gen_kwargs = dict(
            speaker_mode=speaker_mode,
            speaker_audio=wav,
            speaker_sample_rate=int(ref_sr),
            speaker_cfg_scale=float(speaker_cfg_scale),
            temperature=float(temperature),
            topk=int(top_k),
            top_p=float(top_p),
            repetition_penalty=float(repetition_penalty),
            max_new_frames=int(max_new_frames),
            text_normalization=True,
        )
        if seed >= 0:
            gen_kwargs["seed"] = int(seed)

        with torch.inference_mode():
            try:
                result = model.generate(target_text, language=lang_code, **gen_kwargs)
            except (TypeError, ValueError, KeyError, RuntimeError):
                short_lang = lang_code.split("_")[0]
                if short_lang != lang_code:
                    try:
                        result = model.generate(target_text, language=short_lang, **gen_kwargs)
                    except Exception:
                        fallback_kwargs = {**gen_kwargs, "text_normalization": False}
                        result = model.generate(target_text, language="en_us", **fallback_kwargs)
                else:
                    fallback_kwargs = {**gen_kwargs, "text_normalization": False}
                    try:
                        result = model.generate(target_text, language="en_us", **fallback_kwargs)
                    except Exception:
                        fallback_kwargs.pop("text_normalization", None)
                        result = model.generate(target_text, **fallback_kwargs)

        audio, sr = unpack_audio(result, model)
        output_path = "output_cloned_ZONOS3-Mini_CoinNoin.wav"
        sf.write(output_path, audio, sr)

        print(f"✅ Cloning complete! Saved to {output_path} ({sr} Hz)")
        display(Audio(output_path))
    finally:
        if os.path.exists(ref_path):
            os.remove(ref_path)